# Week 7 — Neural Language Models: Practical Tasks

This notebook covers Practical Task 1 (Text Prediction System) and Practical Task 3 (TensorFlow NLP Exercise). Practical Task 2 (the 2-4 page written report) is provided as a separate Word document.

In [ ]:
!pip install tensorflow pdfplumber --quiet

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.utils import to_categorical
import numpy as np
import re

print('TensorFlow version:', tf.__version__)
print('Libraries imported successfully.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 59.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
TensorFlow version: 2.20.0
Libraries imported successfully.


In [ ]:
from google.colab import files
import pdfplumber

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

text_data = ''
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages[:60]:
        page_text = page.extract_text()
        if page_text:
            text_data += page_text + ' '

text_data = text_data.lower()
text_data = re.sub(r'[^a-z\s.]', ' ', text_data)
text_data = re.sub(r'\s+', ' ', text_data).strip()

print(f'Extracted and cleaned {len(text_data)} characters from the CBK report.')

Saving 1190916865_2024 Annual Report.pdf to 1190916865_2024 Annual Report.pdf
Extracted and cleaned 100058 characters from the CBK report.


In [ ]:
# ============================
# TEXT PREDICTION SYSTEM (CBK REPORT)
# ============================

# 1. IMPORT LIBRARIES
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


# ============================
# 1. CREATE A SMALL DATASET
# ============================
# Assume text_data contains a CBK report (monetary policy, inflation, etc.)

sentences = re.split(r'(?<=[.])\s+', text_data)

# Keep only meaningful sentences (5 to 20 words)
sentences = [s.strip() for s in sentences if 5 <= len(s.split()) <= 20]

# Limit dataset size
dataset_sentences = sentences[:200]

print(f'Dataset built from {len(dataset_sentences)} CBK report sentences.\n')

print('Sample sentences:')
for s in dataset_sentences[:3]:
    print(f'  - {s}')


# ============================
# 2. TOKENIZE TEXT
# ============================

tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset_sentences)

# Vocabulary size (unique words)
vocab_size = len(tokenizer.word_index) + 1

print(f'\nVocabulary size: {vocab_size} unique words\n')

print('Sample word index mapping:')
for word, idx in list(tokenizer.word_index.items())[:10]:
    print(f'  {word}: {idx}')


# ============================
# 3. CREATE TRAINING DATA
# ============================

input_sequences = []

# Convert sentences into n-gram sequences
for sentence in dataset_sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to equal length
max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

# Split into features (X) and labels (y)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode labels
y = to_categorical(y, num_classes=vocab_size)

print(f'\nTotal training sequences created: {len(input_sequences)}')
print(f'Max sequence length: {max_seq_len}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')


# ============================
# 4. BUILD NEURAL NETWORK MODEL
# ============================

model = Sequential([
    Embedding(vocab_size, 32, input_length=max_seq_len - 1),  # Word embedding layer
    LSTM(64),                                                # LSTM for sequence learning
    Dense(vocab_size, activation='softmax')                  # Output layer (next word prediction)
])

# Compile model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Show model structure
model.summary()


# ============================
# 5. TRAIN THE MODEL
# ============================

history = model.fit(X, y, epochs=50, verbose=1)

print(f'\nFinal training accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Final training loss:     {history.history["loss"][-1]:.4f}')


# ============================
# 6. NEXT WORD PREDICTION
# ============================

def predict_next_word(seed_text, model, tokenizer, max_seq_len):
    # Convert text to sequence
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # Pad sequence
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

    # Predict probabilities for all words
    predicted_probs = model.predict(token_list, verbose=0)[0]

    # Get highest probability word
    predicted_index = np.argmax(predicted_probs)
    confidence = predicted_probs[predicted_index]

    # Convert index back to word
    predicted_word = ''
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            predicted_word = word
            break

    return predicted_word, confidence


# ============================
# 7. TEST THE MODEL
# ============================

test_phrases = [
    'the central bank',
    'inflation declined',
    'monetary policy',
    'the kenya shilling'
]

print('\nNEXT-WORD PREDICTIONS')
print('=' * 50)

for phrase in test_phrases:
    word, conf = predict_next_word(phrase, model, tokenizer, max_seq_len)
    print(f'"{phrase}" -> predicted next word: "{word}"  (confidence: {conf:.2%})')

Dataset built from 200 CBK report sentences.

Sample sentences:
  - vi statement by the chairman of the board of directors..............................................................vii board of directors.............................................................................................................
  - ix members of the monetary policy committee................................................................................x senior management....................................................................................................................
  - dhow central securities depository dhowcsd ..................................................................
Vocabulary size: 571 unique words

Sample of word index:
  the: 1
  and: 2
  in: 3
  of: 4
  to: 5
  percent: 6
  by: 7
  growth: 8
  a: 9
  fy: 10
Total training sequences created: 1704
Max sequence length: 21
X shape: (1704, 20)
y shape: (1704, 571)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.0446 - loss: 5.9803
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0434 - loss: 5.4825
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0393 - loss: 5.4164
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0446 - loss: 5.3672
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0516 - loss: 5.3060
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0558 - loss: 5.2418
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0540 - loss: 5.1809
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0646 - loss: 5.1186
Epoch 9/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0704 - loss: 5.0672
Epoch 10/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0728 - loss: 5.0123
Epoch 11/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0810 - loss: 4.9587
Epoch 12/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy:

## Practical Task 3 — TensorFlow NLP Exercise

**Requirements:**
- Accept text input
- Tokenize the text
- Display word indices
- Convert text into sequences

In [ ]:
user_text = input('Type a sentence about the CBK report: ')

exercise_tokenizer = Tokenizer()
exercise_tokenizer.fit_on_texts([user_text])

print()
print('YOUR TEXT:')
print(user_text)
print()
print('WORD INDICES:')
print(exercise_tokenizer.word_index)
print()
print('TEXT CONVERTED TO SEQUENCE:')
print(exercise_tokenizer.texts_to_sequences([user_text]))

Type a sentence about the CBK report: inflation rate in 2024

YOUR TEXT:
inflation rate in 2024

WORD INDICES:
{'inflation': 1, 'rate': 2, 'in': 3, '2024': 4}

TEXT CONVERTED TO SEQUENCE:
[[1, 2, 3, 4]]
